## Modular Pipeline Integration Test

In [1]:
# Standard imports
import pandas as pd
import numpy as np
import torch
import os
import json
import sys
from pathlib import Path
from torch.utils.data import DataLoader
from torchvision import transforms
from PIL import Image


# Add project root to path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(project_root)


# Pipeline imports
from src.pipeline.data_ingestion import load_csv, build_image_label_df
from src.pipeline.preprocessing import preprocess_data
from src.pipeline.feature_engineering import generate_embeddings
from src.pipeline.datasets import ProductDataset
from src.pipeline.evaluation import evaluate_model
from src.pipeline.inference import predict
from src.initialization import load_environment

# Utility imports
from src.utils.vector_db_utils import VectorDBManager
from src.utils.data_cleaning import clean_stock_code
from src.utils.model_loader import load_model
from src.utils.generate_cnn_csv import generate_final_cnn_training_data

# Script imports
from src.scripts.train_cnn_from_scratch import main

# Model imports
from src.models.cnn_model import CNNModel

# calls
load_environment()
generate_final_cnn_training_data()

INFO:src.utils.model_loader:Loading model from: c:\Users\don\dev\ds_test\models\best_model.pth
INFO:src.utils.model_loader:Loading label mapping from: c:\Users\don\dev\ds_test\src\data\label_mapping.json
INFO:src.utils.model_loader:Loaded label mapping: {'0': 'LUNCH BAG WOODLAND', '1': 'REX CASH+CARRY JUMBO SHOPPER', '2': 'JUMBO STORAGE BAG SUKI', '3': '6 RIBBONS RUSTIC CHARM', '4': 'CHOCOLATE HOT WATER BOTTLE', '5': 'RETROSPOT TEA SET CERAMIC 11 PC', '6': 'LUNCH BAG PINK POLKADOT', '7': 'REGENCY CAKESTAND 3 TIER', '8': 'ALARM CLOCK BAKELIKE RED', '9': 'SPOTTY BUNTING'}


[DEBUG] Current working directory: c:\Users\don\dev\ds_test\notebooks
[DEBUG] Model path (as passed): c:\Users\don\dev\ds_test\models\best_model.pth
[DEBUG] Model path (absolute): c:\Users\don\dev\ds_test\models\best_model.pth


INFO:src.utils.model_loader:Initialized model with 10 classes
INFO:src.utils.model_loader:Loaded state dict with keys: odict_keys(['conv1.0.weight', 'conv1.0.bias', 'conv1.1.weight', 'conv1.1.bias', 'conv1.1.running_mean', 'conv1.1.running_var', 'conv1.1.num_batches_tracked', 'conv2.0.weight', 'conv2.0.bias', 'conv2.1.weight', 'conv2.1.bias', 'conv2.1.running_mean', 'conv2.1.running_var', 'conv2.1.num_batches_tracked', 'conv3.0.weight', 'conv3.0.bias', 'conv3.1.weight', 'conv3.1.bias', 'conv3.1.running_mean', 'conv3.1.running_var', 'conv3.1.num_batches_tracked', 'conv4.0.weight', 'conv4.0.bias', 'conv4.1.weight', 'conv4.1.bias', 'conv4.1.running_mean', 'conv4.1.running_var', 'conv4.1.num_batches_tracked', 'fc.1.weight', 'fc.1.bias', 'fc.4.weight', 'fc.4.bias'])
INFO:src.utils.model_loader:Successfully loaded state dict into model
INFO:src.utils.model_loader:Model set to evaluation mode
INFO:src.utils.model_loader:Test input shape: torch.Size([1, 3, 224, 224])
INFO:src.utils.model_loade

INFO: Model and transform initialized successfully


INFO:src.initialization:Model and transform initialized successfully


### Data Ingestion & Cleaning
Load raw data from CSV, drop duplicates, apply cleaning/ preprocessing

In [ ]:
raw_df = load_csv('../src/data/dataset/dataset.csv')
raw_df = raw_df.drop_duplicates()

# Clean and preprocess
cleaned_df = preprocess_data(raw_df)

cleaned_df = cleaned_df.drop_duplicates(subset='StockCode', keep='last')

cleaned_df.to_csv('../src/data/dataset/cleaned/cleaned_cnn.csv')

### Feature Engineering & Embedding Generation
Generate feature embeddings for the deduplicated product data (for vector DB)

In [ ]:
api_key = os.getenv('PINECONE_API_KEY')

vector_db_manager = VectorDBManager(api_key=api_key)
vector_db_manager.initialize()

embeddings = generate_embeddings(cleaned_df, vector_db_manager)

embeddings_df = pd.DataFrame(np.array(embeddings))
embeddings_df.head()

### Model Training
Training the model with scraped data

In [ ]:
# Run the training script
try:
    history = main(
        project_root_str=project_root,
        batch_size=32,
        num_epochs=50,
        learning_rate=0.001
    )
    display("\nTraining completed successfully!")
    display(f"Best validation accuracy: {history['best_val_acc']:.2f}%")
except Exception as e:
    display(f"Error occurred: {str(e)}")
    raise

### Evaluation

In [ ]:
# Load model after training
model_path = "models/best_model.pth"
label_mapping_path = "src/data/label_mapping.json"
model, label_mapping = load_model(model_path=model_path, label_mapping_path=label_mapping_path)
if model:
    display("Model loaded")

INFO:src.utils.model_loader:Loading model from: c:\Users\don\dev\ds_test\models\best_model.pth
INFO:src.utils.model_loader:Loading label mapping from: c:\Users\don\dev\ds_test\src\data\label_mapping.json
INFO:src.utils.model_loader:Loaded label mapping: {'0': 'LUNCH BAG WOODLAND', '1': 'REX CASH+CARRY JUMBO SHOPPER', '2': 'JUMBO STORAGE BAG SUKI', '3': '6 RIBBONS RUSTIC CHARM', '4': 'CHOCOLATE HOT WATER BOTTLE', '5': 'RETROSPOT TEA SET CERAMIC 11 PC', '6': 'LUNCH BAG PINK POLKADOT', '7': 'REGENCY CAKESTAND 3 TIER', '8': 'ALARM CLOCK BAKELIKE RED', '9': 'SPOTTY BUNTING'}
INFO:src.utils.model_loader:Initialized model with 10 classes
INFO:src.utils.model_loader:Loaded state dict with keys: odict_keys(['conv1.0.weight', 'conv1.0.bias', 'conv1.1.weight', 'conv1.1.bias', 'conv1.1.running_mean', 'conv1.1.running_var', 'conv1.1.num_batches_tracked', 'conv2.0.weight', 'conv2.0.bias', 'conv2.1.weight', 'conv2.1.bias', 'conv2.1.running_mean', 'conv2.1.running_var', 'conv2.1.num_batches_tracked', 

[DEBUG] Current working directory: c:\Users\don\dev\ds_test\notebooks
[DEBUG] Model path (as passed): c:\Users\don\dev\ds_test\models\best_model.pth
[DEBUG] Model path (absolute): c:\Users\don\dev\ds_test\models\best_model.pth


INFO:src.utils.model_loader:Successfully loaded state dict into model
INFO:src.utils.model_loader:Model set to evaluation mode
INFO:src.utils.model_loader:Test input shape: torch.Size([1, 3, 224, 224])
INFO:src.utils.model_loader:Raw model output shape: torch.Size([1, 10])
INFO:src.utils.model_loader:Raw model output: [-1.3610378503799438, 0.3127504587173462, 2.4162747859954834, 7.63803243637085, -0.5466543436050415, -13.847339630126953, 8.740796089172363, -7.656243324279785, 6.020637035369873, 7.631875514984131]
INFO:src.utils.model_loader:Probabilities: [2.3703594706603326e-05, 0.00012639547639992088, 0.001035811030305922, 0.1918938159942627, 5.351758227334358e-05, 8.95535520739621e-11, 0.5780762434005737, 4.373623596620746e-08, 0.03807457536458969, 0.1907159686088562]
INFO:src.utils.model_loader:Test prediction: Class 6 (StockCode: 6, Label: LUNCH BAG PINK POLKADOT) with confidence 0.5781


'Model loaded'

In [2]:
with open('../models/stockcode_to_index.json') as f:
    stockcode_to_index = json.load(f)

images_root = r'../model_test'
df = build_image_label_df(images_root, stockcode_to_index)

In [3]:
# Prepare the dataset and DataLoader
image_paths = df['image_path']  # List of image paths for each row in train_df
labels = df['label']       # List of class indices for each row in train_df

train_dataset = ProductDataset(image_paths, labels, transform=None)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=False)

# Load the trained model
num_classes = len(set(labels))
model = CNNModel(num_classes)
model.load_state_dict(torch.load('../models/best_model.pth', map_location='cpu'))
model.eval()

# Evaluate
accuracy, metrics = evaluate_model(model, train_loader, labels)
display(f"Training set accuracy: {accuracy}")
display(f"Metrics: {metrics}")

'Training set accuracy: 0.7'

"Metrics: {'accuracy': 0.7}"

### Inference

In [4]:
model_path = '../models/best_model.pth'
num_classes = 10

# Load the model
model = CNNModel(num_classes)
model.load_state_dict(torch.load(model_path, map_location='cpu'))
model.eval()

# Define transforms
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load and preprocess the image
img_path = '../model_test/1.jpg'
image = Image.open(img_path).convert('RGB')
input_tensor = transform(image)

# Run inference
predicted_class_idx = predict(model, input_tensor)
display(f"Predicted class index: {predicted_class_idx}")

'Predicted class index: 8'